In [4]:
import pandas as pd

H3N2_42 = pd.read_excel('/data/chenyihao/fluProfiler_source/data/raw/data4model(H3N2-42).xlsx', index_col=False)

In [7]:
H3N2_42.to_csv('/data/chenyihao/fluProfiler_source/data/raw/data4model(H3N2-42).csv', index=False)

In [9]:
H3N2_42.columns

Index(['serumName', 'serumPassage', 'serumPassCat', 'serumDate', 'serumType',
       'ferret', 'virusName', 'virusPassage', 'virusPassCat', 'virusDate',
       'virusType', 'dataSource', 'serumIslID', 'serumMatchedPass',
       'virusIslID', 'virusMatchedPass', 'HI_Dist', 'serumHA', 'serumNA',
       'virusHA', 'virusNA'],
      dtype='object')

In [24]:
H3N2_42_filt1 = H3N2_42[['serumHA', 'serumNA', 'virusHA', 'virusNA', 'serumPassCat', 'virusPassCat', 
                         'serumName', 'virusName', 'virusDate', 'virusIslID', 'serumType', 'HI_Dist']].copy()
H3N2_42_filt1.columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 
                         'serumName', 'virusName', 'virusDate', 'virusIslID', 'serumType', 'HI_Dist']
unique_HA = pd.concat([H3N2_42_filt1['seq_a'], H3N2_42_filt1['seq_c']]).unique().tolist()
unique_NA = pd.concat([H3N2_42_filt1['seq_b'], H3N2_42_filt1['seq_d']]).unique().tolist()

In [26]:
HA_dict = {key: value for key, value in zip(unique_HA,['HA_' + str(i) for i in range(len(unique_HA))])}
NA_dict = {key: value for key, value in zip(unique_NA,['NA_' + str(i) for i in range(len(unique_NA))])}

In [28]:
H3N2_42_filt1['seq_id_a'] = H3N2_42_filt1['seq_a'].map(HA_dict)
H3N2_42_filt1['seq_id_b'] = H3N2_42_filt1['seq_b'].map(NA_dict)
H3N2_42_filt1['seq_id_c'] = H3N2_42_filt1['seq_c'].map(HA_dict)
H3N2_42_filt1['seq_id_d'] = H3N2_42_filt1['seq_d'].map(NA_dict)


In [31]:
H3N2_42_filt2 = H3N2_42_filt1[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d','seq_a', 'seq_b', 'seq_c', 'seq_d', 
                               'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'virusDate', 'virusIslID', 'serumType', 'HI_Dist']].copy()

In [ ]:
H3N2_42_filt2.to_csv('../../data/processed/H3N2_42.csv', index=False)

In [66]:
from torch.utils.data import Dataset, DataLoader
import sys
sys.path.append('../')
from utilities import load_embedding
import torch

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask


In [62]:
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
new_columns = ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d','seq_a', 'seq_b', 'seq_c', 'seq_d', 
               'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'virusIslID', 'HI_Dist']
dataframe = H3N2_42_filt2.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'seq_id_c': 'first', 'seq_id_d': 'first',
                                                      'serumName': 'first', 'virusName': 'first', 'virusIslID': 'first','HI_Dist': 'mean'}).reset_index()
Crick_all_final = dataframe[new_columns]

In [63]:
Crick_all_final['serumPassCat'] = Crick_all_final['serumPassCat'].str.replace('CELL', '<CELL>').str.replace('EGG', '<EGG>')
Crick_all_final['virusPassCat'] = Crick_all_final['virusPassCat'].str.replace('CELL', '<CELL>').str.replace('EGG', '<EGG>')
Crick_all_final['label'] = Crick_all_final['HI_Dist'] 

In [67]:
test_dataset = fluProfiler_Dataset(Crick_all_final)
test_dataloader = DataLoader(test_dataset, batch_size=200, shuffle=False)

In [ ]:
import os
from tqdm import tqdm

embedding_df = Crick_all_final
# load embedding
sequence_names = pd.concat([embedding_df['seq_id_a'],embedding_df['seq_id_b'],
                            embedding_df['seq_id_c'],embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding_42", files=sequence_names)
# embeddings = [emb.to(device) for emb in embeddings]
emb_dict = dict(zip(IDs, embeddings))

Loading tensor: 100%|██████████| 409/409 [00:15<00:00, 27.17file/s]


In [69]:
device = torch.device('cuda:1')
model = torch.load('/data/chenyihao/fluProfiler_source/model/model_001.pth', weights_only=False, map_location=device)

In [73]:
import torch.nn.functional as F
from utilities import print_exams

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                     matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                     matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                     labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

NameError: name 'print_exams' is not defined

In [74]:
from utilities import print_exams
print_exams(reference_ls, prediction_ls)

MAE:  1.122346418690618
MSE:  2.3977337721622636
pearson correlation:  PearsonRResult(statistic=0.5026790776882274, pvalue=7.907866032133507e-145)
spearman correlation:  SignificanceResult(statistic=0.5393759851743924, pvalue=1.4040462518399716e-170)
R2_score:  -2.3005941914417045
